In [ ]:
!pip install -q open_clip_torch

In [ ]:
import torch
import open_clip
import requests
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32-quickgelu",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32-quickgelu")
model = model.to(device)
model.eval()

print("设备:", device)
print("模型加载完成")

In [ ]:
%cd /content

# 下载图片，约 1GB
!wget -O val2017.zip -q --show-progress http://images.cocodataset.org/zips/val2017.zip

# 下载标注
!wget -O annotations_trainval2017.zip -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# 解压；-o 表示覆盖已有文件
!unzip -o -q val2017.zip
!unzip -o -q annotations_trainval2017.zip

# 检查图片数量
print("图片数量:")
!find /content/val2017 -maxdepth 1 -name "*.jpg" | wc -l

# 检查标注文件
print("标注文件:")
!ls -lh /content/annotations/captions_val2017.json

In [ ]:
import os
import json
from PIL import Image

# 1. 读取 COCO 标注
with open("/content/annotations/captions_val2017.json", "r", encoding="utf-8") as f:
    coco_data = json.load(f)

# 图片编号 -> 图片信息
images_by_id = {
    img["id"]: img
    for img in coco_data["images"]
}

# 图片编号 -> 多条描述
captions_by_image = {}
for ann in coco_data["annotations"]:
    image_id = ann["image_id"]
    caption = ann["caption"]
    captions_by_image.setdefault(image_id, []).append(caption)

# 取前 100 张有描述的图片
selected_image_ids = list(captions_by_image.keys())[:100]

images = []
captions = []
valid_image_ids = []
missing_files = []

# 2. 加载前 100 张图片
for image_id in selected_image_ids:
    image_info = images_by_id[image_id]
    image_path = f"/content/val2017/{image_info['file_name']}"

    if not os.path.exists(image_path):
        missing_files.append(image_path)
        continue

    try:
        # 打开图片并转成 RGB
        image = Image.open(image_path).convert("RGB")

        # 图片预处理
        images.append(preprocess(image))

        # 第一版先只用每张图片的第一条描述
        captions.append(captions_by_image[image_id][0])

        valid_image_ids.append(image_id)

    except Exception as e:
        print("加载失败:", image_path)
        print("原因:", e)

print("成功加载图片:", len(images))
print("缺失图片:", len(missing_files))

# 3. 图片打包成张量
# 形状：[图片数量, 3, 224, 224]
image_input = torch.stack(images).to(device)

# 4. 文字变 token
text_input = tokenizer(captions).to(device)

# 5. 编码图片和文字
with torch.no_grad():
    image_features = model.encode_image(image_input)
    text_features = model.encode_text(text_input)

    # 归一化
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # 相似度矩阵
    # 行：图片
    # 列：文字
    similarity = image_features @ text_features.T

# 6. 每张图片选择最匹配的文字
predicted_indices = similarity.argmax(dim=1)

# 正确答案就是每张图片自己的索引
num_samples = len(captions)
correct_indices = torch.arange(num_samples, device=device)

# 7. 计算 Top-1 准确率
correct_count = (predicted_indices == correct_indices).sum().item()
top1_accuracy = correct_count / num_samples

print("\n图片数量:", num_samples)
print("Top-1 准确率:", top1_accuracy)
print("Top-1 正确数量:", correct_count, "/", num_samples)

# 8. 计算 Recall@1/5/10
print("\nRecall 指标:")

for k in [1, 5, 10]:
    # 每张图片取相似度最高的 k 个文字
    topk_indices = similarity.topk(k, dim=1).indices

    # 正确答案是否出现在前 k 个结果中
    correct_k = (
        topk_indices == correct_indices.unsqueeze(1)
    ).any(dim=1).sum().item()

    recall_k = correct_k / num_samples
    print(f"Recall@{k}: {recall_k:.4f}")

# 9. 查看前 5 张图片的结果
print("\n前 5 张图片的预测结果:")

for i in range(min(5, num_samples)):
    pred_index = predicted_indices[i].item()
    status = "正确" if pred_index == i else "错误"

    print(f"\n图片 {i + 1}")
    print("正确描述:", captions[i])
    print("预测描述:", captions[pred_index])
    print("结果:", status)